# Limpieza de datos - Cuentas ILMS Koha

Auditoría de calidad sobre el reporte de cuentas del ILMS Koha.  
**Propósito:** diagnóstico, no mutación. Produce CSVs accionables para que el equipo DBA corrija en Koha.

---
**Acceso a los datos:** los CSV originales están en una carpeta de Google Drive restringida a la comunidad UdeA. 

**Para ejecutar desde cero:** `Runtime → Run all`

## 0. Parámetros de configuración



In [ ]:
#  MODO DE CARGA
# 'drive'   monta Google Drive 
# 'local'  
MODO = 'drive'

# Ruta dentro de mi Drive 

RUTA_DRIVE = '/content/drive/MyDrive/Biblioteca/limpieza-cuentas-koha/'

# CSV
FILE_REPORTE  = 'Reporte-cuentas-ILMS-Koha.csv'
FILE_CATEGORIAS = 'Tabla_categorias_cuentas-ILMS_Koha.csv'

# Carpeta de salida 
OUTPUT_DIR = 'output'

# Dominios de correo considerados institucionales 
DOMINIOS_INSTITUCIONALES = ['udea.edu.co']

# Categorías que DEBEN tener correo institucional
CATS_REQUIEREN_CORREO_INST = ['DOCEN', 'DOCAT', 'ESPRE', 'ESPOS', 'EGRES', 'INVES', 'EOFIC']

## 1. Setup

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import unicodedata

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup OK')

## 2. Carga de datos

In [ ]:
if MODO == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    base = RUTA_DRIVE
elif MODO == 'local':
    base = 'data/'
else:
    raise ValueError(f'MODO desconocido: {MODO!r}. Usa "drive" o "local".')

df  = pd.read_csv(base + FILE_REPORTE,    dtype=str, keep_default_na=False)
cat = pd.read_csv(base + FILE_CATEGORIAS, dtype=str, keep_default_na=False)

# Reemplazar 'NULL' por NaN
df.replace('NULL', np.nan, inplace=True)
cat.replace('NULL', np.nan, inplace=True)

TOTAL = len(df)
print(f'Registros cargados: {TOTAL:,}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

## 3. Normalización

In [ ]:
def norm_str(s):
    """strip + lower + colapsar espacios internos"""
    if pd.isna(s):
        return np.nan
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def norm_cedula(s):
    """strip + quitar puntos, comas y espacios"""
    if pd.isna(s):
        return np.nan
    return re.sub(r'[.,\s]', '', str(s).strip())

def norm_email(s):
    if pd.isna(s):
        return np.nan
    return str(s).strip().lower()

def tiene_espacios_extremos(s):
    if pd.isna(s):
        return False
    return str(s) != str(s).strip()

def tiene_chars_no_imprimibles(s):
    if pd.isna(s):
        return False
    return any(unicodedata.category(c) in ('Cc', 'Cf') for c in str(s))

df['_cedula']  = df['cardnumber'].apply(norm_cedula)
df['_email']   = df['email'].apply(norm_email)
df['_emailpro'] = df['emailpro'].apply(norm_email)
df['_surname'] = df['surname'].apply(norm_str)
df['_firstname'] = df['firstname'].apply(norm_str)
df['_userid']  = df['userid'].apply(norm_str)

print('Normalización aplicada.')

## 4. Detección de anomalías

Cada sección genera un CSV en `output/` (en el entorno de desarrollo) con las filas afectadas y una columna `motivo`.

In [ ]:
resultados = []  # acumulador para el resumen final

def guardar(mask, nombre, motivo):
    """Filtra, agrega columna motivo y guarda CSV."""
    subset = df[mask].copy()
    subset['motivo'] = motivo
    cols_out = [c for c in df.columns if not c.startswith('_')] + ['motivo']
    ruta = f'{OUTPUT_DIR}/{nombre}.csv'
    subset[cols_out].to_csv(ruta, index=False)
    n = len(subset)
    pct = n / TOTAL * 100
    print(f'   {nombre}: {n:,} registros ({pct:.2f}%)')
    resultados.append({'anomalia': nombre, 'conteo': n, 'porcentaje': round(pct, 2), 'archivo': ruta})
    return mask

### 4.1 Sin cédula

In [ ]:
mask_sin_cedula = df['_cedula'].isna() | (df['_cedula'] == '')
guardar(mask_sin_cedula, '01_sin_cedula', 'cardnumber nulo o vacío')

### 4.2 Cédulas repetidas

In [ ]:
cedulas_validas = df['_cedula'].notna() & (df['_cedula'] != '')
duplicadas = df.loc[cedulas_validas, '_cedula'].duplicated(keep=False)
mask_cedula_dup = cedulas_validas & duplicadas
guardar(mask_cedula_dup, '02_cedulas_repetidas', 'cardnumber duplicado')

### 4.3 Cédula con X al inicio o al final

In [ ]:
mask_cedula_x = df['_cedula'].notna() & df['_cedula'].str.match(r'^[xX]|[xX]$', na=False)
guardar(mask_cedula_x, '03_cedula_con_X', 'cardnumber comienza o termina con X')

### 4.4 Correos repetidos

In [ ]:
emails_validos = df['_email'].notna() & (df['_email'] != '')
dup_email = df.loc[emails_validos, '_email'].duplicated(keep=False)
mask_email_dup = emails_validos & dup_email
guardar(mask_email_dup, '04_correos_repetidos', 'email duplicado')

### 4.5 Cuentas genéricas (con lo que se hasta ahora)

In [ ]:
PALABRAS_GENERICAS = ['equipo', 'biblioteca', 'test', 'prueba', 'admin', 'demo',
                      'sistema', 'generic', 'null', 'usuario', 'user']

# surname == firstname
cond_sn_eq_fn = (
    df['_surname'].notna() & df['_firstname'].notna() &
    (df['_surname'] == df['_firstname'])
)
# userid == cardnumber (normalizado)
cond_userid_cedula = (
    df['_userid'].notna() & df['_cedula'].notna() &
    (df['_userid'] == df['_cedula'])
)
# nombre o apellido contiene palabra genérica
pat_generica = '|'.join(PALABRAS_GENERICAS)
cond_nombre_gen = (
    df['_surname'].str.contains(pat_generica, na=False) |
    df['_firstname'].str.contains(pat_generica, na=False)
)
# cardnumber no numérico en categorías que no son STAFF
cats_no_staff = ~df['categorycode'].isin(['STAFF', 'ADMIN', '_IPWV', 'BUSYS', 'ESTAD'])
cedula_no_numerica = df['_cedula'].notna() & ~df['_cedula'].str.match(r'^\d+$', na=False)
cond_cedula_alfanumerica = cats_no_staff & cedula_no_numerica

mask_genericas = cond_sn_eq_fn | cond_userid_cedula | cond_nombre_gen | cond_cedula_alfanumerica

# Construir motivo detallado por fila
motivos = []
for _, row in df[mask_genericas].iterrows():
    m = []
    if cond_sn_eq_fn[row.name]: m.append('surname==firstname')
    if cond_userid_cedula[row.name]: m.append('userid==cardnumber')
    if cond_nombre_gen[row.name]: m.append('nombre genérico')
    if cond_cedula_alfanumerica[row.name]: m.append('cédula no numérica en categoría no-STAFF')
    motivos.append('; '.join(m))

sub_gen = df[mask_genericas].copy()
sub_gen['motivo'] = motivos
cols_out = [c for c in df.columns if not c.startswith('_')] + ['motivo']
sub_gen[cols_out].to_csv(f'{OUTPUT_DIR}/05_cuentas_genericas.csv', index=False)
n = len(sub_gen)
pct = n / TOTAL * 100
print(f'   05_cuentas_genericas: {n:,} registros ({pct:.2f}%)')
resultados.append({'anomalia': '05_cuentas_genericas', 'conteo': n, 'porcentaje': round(pct, 2), 'archivo': f'{OUTPUT_DIR}/05_cuentas_genericas.csv'})

### 4.6 Cuentas de préstamo interbibliotecario

In [ ]:
mask_prein = df['categorycode'] == 'PREIN'
guardar(mask_prein, '06_prestamo_interbibliotecario', 'categorycode == PREIN')

### 4.7 Campos nulos (nombre, correo, userid)

In [ ]:
campos_criticos = {
    'surname':   'apellido (surname) nulo o vacío',
    'firstname': 'nombre (firstname) nulo o vacío',
    'email':     'correo (email) nulo o vacío',
    'userid':    'userid nulo o vacío',
}
for campo, motivo in campos_criticos.items():
    col_norm = f'_{campo}'
    if col_norm in df.columns:
        mask = df[col_norm].isna() | (df[col_norm].str.strip() == '')
    else:
        mask = df[campo].isna() | (df[campo].str.strip() == '')
    guardar(mask, f'07_nulo_{campo}', motivo)

# Consolidado: al menos un campo crítico nulo
mask_any_null = (
    (df['_surname'].isna()   | (df['_surname'].str.strip()   == '')) |
    (df['_firstname'].isna() | (df['_firstname'].str.strip() == '')) |
    (df['_email'].isna()     | (df['_email'].str.strip()     == '')) |
    (df['_userid'].isna()    | (df['_userid'].str.strip()    == ''))
)
guardar(mask_any_null, '07_campos_nulos_consolidado', 'al menos un campo crítico nulo')

### 4.8 Empleados CIS

> criterio pendiente

In [ ]:
def is_cis(row):
    # #
    return False

mask_cis = df.apply(is_cis, axis=1)
print(f'  ⏸ 08_empleados_CIS: criterio pendiente — {mask_cis.sum()} registros actualmente')
# guardar(mask_cis, '08_empleados_CIS', 'empleado CIS')  # descomentar cuando le pregunte a Andrés

---
## 5. Otras anomalías

### 5.1 Correos con formato inválido

In [ ]:
EMAIL_RE = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
emails_presentes = df['_email'].notna() & (df['_email'] != '')
mask_email_invalido = emails_presentes & ~df['_email'].str.match(EMAIL_RE, na=False)
guardar(mask_email_invalido, 'X1_correo_formato_invalido', 'email con formato inválido')

### 5.2 Correos no institucionales en categorías que deberían tenerlo

In [ ]:
def es_institucional(email):
    if pd.isna(email) or email == '':
        return True  # los nulos ya los reporté en otro apartado
    return any(email.endswith('@' + d) for d in DOMINIOS_INSTITUCIONALES)

en_cat_institucional = df['categorycode'].isin(CATS_REQUIEREN_CORREO_INST)
mask_correo_no_inst = en_cat_institucional & ~df['_email'].apply(es_institucional)
guardar(mask_correo_no_inst, 'X2_correo_no_institucional', 'correo no institucional en categoría que lo requiere')

### 5.3 userid duplicado

In [ ]:
userid_validos = df['_userid'].notna() & (df['_userid'] != '')
dup_userid = df.loc[userid_validos, '_userid'].duplicated(keep=False)
mask_userid_dup = userid_validos & dup_userid
guardar(mask_userid_dup, 'X3_userid_duplicado', 'userid duplicado')

### 5.4 Cédulas con longitud atípica (< 6 o > 10 dígitos)

In [ ]:
cedulas_numericas = df['_cedula'].notna() & df['_cedula'].str.match(r'^\d+$', na=False)
longitud = df.loc[cedulas_numericas, '_cedula'].str.len()
mask_longitud = cedulas_numericas & ((longitud < 6) | (longitud > 10))
guardar(mask_longitud, 'X4_cedula_longitud_atipica', 'cédula numérica con longitud < 6 o > 10')

### 5.5 Espacios al inicio/final en campos clave

In [ ]:
campos_espacios = ['cardnumber', 'surname', 'firstname', 'email', 'userid']
mask_espacios = pd.Series(False, index=df.index)
for c in campos_espacios:
    mask_espacios = mask_espacios | df[c].apply(tiene_espacios_extremos)
guardar(mask_espacios, 'X5_espacios_extremos', 'campo con espacio al inicio o final')

### 5.6 categorycode sin correspondencia en tabla de categorías (por si hay FK rota)

In [ ]:
cat_codes = set(cat['categorycode'].str.strip().unique())
mask_fk = ~df['categorycode'].str.strip().isin(cat_codes)
guardar(mask_fk, 'X6_categorycode_invalido', 'categorycode no existe en tabla de categorías')

---
## 6. Resumen 

In [ ]:
resumen = pd.DataFrame(resultados)
resumen = resumen.sort_values('conteo', ascending=False).reset_index(drop=True)
resumen.to_csv(f'{OUTPUT_DIR}/RESUMEN.csv', index=False)

print(f'\nTotal cuentas analizadas: {TOTAL:,}\n')
resumen.style.bar(subset=['conteo'], color='#d65f5f').format({'porcentaje': '{:.2f}%'})

### Distribución de anomalías

### Distribución por categoría de cuenta